# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# --- setup: clone repo in Colab, put work/src on the path, build (or load cached) cutoff frames ---
import os, sys, json, datetime as dt, importlib
REPO_DIR = "/content/FlyRank-ML-Internship"
if os.path.isdir("/content") and not os.environ.get("CAPSTONE_LOCAL"):
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone https://github.com/joannadulay/FlyRank-ML-Internship.git {REPO_DIR}")
    else:
        os.system(f"git -C {REPO_DIR} pull -q")            # always use the latest code and JSON files from GitHub
    os.chdir(REPO_DIR)
    os.system("pip install -q datasets duckdb scikit-learn huggingface_hub pyarrow")
sys.path.insert(0, "work/src")
importlib.invalidate_caches()
import numpy as np, pandas as pd
import capstone_lib as L
os.makedirs(L.OUT_DIR, exist_ok=True)
plan, frames, TOKEN, LOCAL = L.start()
train, val = frames["train"], frames["val"]          # the sealed test frame is not touched in this notebook
print("cutoffs  train:", [str(t) for t in plan["train"]], "| val:", plan["val"])
print("rows     train:", len(train), "| val:", len(val))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 72.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 86.2MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cutoffs  train: ['2026-01-11', '2026-02-08', '2026-03-08', '2026-04-05'] | val: 2026-05-03
rows     train: 237571 | val: 85997


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Three actions, in priority order (a page gets only the first that applies):
1. **review_decline_risk**: top 5% of pages by *predicted decline probability × impressions in the last 28 days* (traffic at stake).
2. **watch_recovery**: top 3% by predicted Recovery probability. Look, but do not rewrite a page that is already bouncing back.
3. **support_growth**: top 3% by predicted Growth probability.

Reason codes come from the page's own recent signals, not from model internals, so a person can check them.

In [2]:
choice = json.load(open("work/outputs/model_choice.json"))
test, final = L.get_final(plan, frames, choice)      # fits the frozen choice on train+val, scores the sealed test cutoff
n = len(test)
test["risk"] = test.p_Decline * test.iB
test["action"] = "no_action"
rank = lambda s: s.rank(ascending=False, method="first")
test.loc[rank(test.p_Growth) <= int(0.03 * n), "action"] = "support_growth"
test.loc[rank(test.p_Recovery) <= int(0.03 * n), "action"] = "watch_recovery"
test.loc[rank(test.risk) <= int(0.05 * n), "action"] = "review_decline_risk"

# reason codes: thresholds learned from TRAINING data only
tr = pd.concat([train, val])
edges = np.unique(np.quantile(tr.pos_B, np.linspace(0, 1, 6))); edges[0], edges[-1] = -np.inf, np.inf
ctr_med = tr.groupby(pd.cut(tr.pos_B, edges), observed=True).ctr_B.median()
band = pd.cut(test.pos_B, edges)
test["ctr_ratio"] = test.ctr_B / band.map(ctr_med).astype(float).replace(0, np.nan)
p90_vol, p90_cv = tr.iB.quantile(.9), tr.cv_B.quantile(.9)
flag = {
    "falling_impressions": test.d_impr <= -0.2, "short_term_slide": test.r7 <= -0.15,
    "position_worsening": test.d_pos >= 1, "ctr_below_peers": test.ctr_ratio < 0.5,
    "high_volume": test.iB >= p90_vol, "volatile": test.cv_B >= p90_cv, "thin_data": test.avail_share_B < 0.75,
}
queue = test[test.action != "no_action"].copy()
queue["reasons"] = [";".join(k for k, v in flag.items() if v.loc[i]) or "model_only" for i in queue.index]
queue = queue.sort_values(["action", "risk"], ascending=[True, False])
print(queue.action.value_counts().to_string(), "\n")
print(queue[["action", "risk", "p_Decline", "p_Recovery", "p_Growth", "iB", "reasons"]].head(10).round(3).to_string())

action
review_decline_risk    4345
support_growth         2607
watch_recovery         2607 

                    action        risk  p_Decline  p_Recovery  p_Growth        iB                                                           reasons
85908  review_decline_risk  224176.534      0.376       0.000     0.018  596368.0                                      short_term_slide;high_volume
2225   review_decline_risk  180388.461      0.910       0.000     0.043  198329.0             short_term_slide;ctr_below_peers;high_volume;volatile
13765  review_decline_risk  173611.568      0.892       0.000     0.045  194709.0                      short_term_slide;ctr_below_peers;high_volume
14270  review_decline_risk  148639.650      0.412       0.000     0.220  360698.0                                                       high_volume
11547  review_decline_risk  118469.864      0.844       0.109     0.000  140443.0  falling_impressions;short_term_slide;ctr_below_peers;high_volume
45647  review_decli

In [3]:
# Does the queue earn its place? Observed outcomes in the sealed month vs doing nothing clever.
base_decline = float((test.y == "Decline").mean())
def share(mask, cls="Decline"): return float((test.loc[mask, "y"] == cls).mean())
rk = rank(test.risk)
naive = rank(-test.d_impr.fillna(0) * test.iB)          # naive: biggest recent fall x volume
naive_r7 = rank(-test.r7.fillna(0) * test.iB)         # naive 2: weakest last-week trend x volume (what the w06 scan found strongest)
slices = [("Top 100 by risk", rk <= 100), ("Top 500 by risk", rk <= 500), ("Top 1,000 by risk", rk <= 1000),
          ("review_decline_risk (top 5%)", test.action == "review_decline_risk"),
          ("Naive: biggest recent fall x volume (top 5%)", naive <= int(0.05 * n)),
          ("Naive: weakest last-week trend x volume (top 5%)", naive_r7 <= int(0.05 * n))]
prec = [dict(name=k, n=int(m.sum()), share_declined=share(m), lift=share(m) / base_decline if base_decline else float("nan")) for k, m in slices]
for r in prec: print(f"{r['name']:<48} n={r['n']:>7,}  declined {r['share_declined']:.1%}  lift {r['lift']:.2f}x")
print(f"\nBase rate of Decline over all test pages: {base_decline:.1%}")
other = dict(watch_recovery_share_recovery=share(test.action == "watch_recovery", "Recovery"), base_recovery=float((test.y == "Recovery").mean()),
             support_growth_share_growth=share(test.action == "support_growth", "Growth"), base_growth=float((test.y == "Growth").mean()))
print({k: round(v, 3) for k, v in other.items()})

Top 100 by risk                                  n=    100  declined 76.0%  lift 1.26x
Top 500 by risk                                  n=    500  declined 80.8%  lift 1.34x
Top 1,000 by risk                                n=  1,000  declined 79.1%  lift 1.31x
review_decline_risk (top 5%)                     n=  4,345  declined 79.2%  lift 1.31x
Naive: biggest recent fall x volume (top 5%)     n=  4,345  declined 75.6%  lift 1.25x
Naive: weakest last-week trend x volume (top 5%) n=  4,345  declined 87.7%  lift 1.45x

Base rate of Decline over all test pages: 60.4%
{'watch_recovery_share_recovery': 0.394, 'base_recovery': 0.063, 'support_growth_share_growth': 0.426, 'base_growth': 0.079}


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [5]:
print("""INTENDED USE AND LIMITS

Who: SEO specialists and content managers deciding which pages to look at first this month.
For what: ordering a review list. It is decision-support, not an instruction to change anything.
Valid for: pages with at least 14 GSC-available days in each of the last two 28-day windows and at least 100 impressions in the latest window. Outside that, the model has seen nothing like the page.
Stops being valid: after a site migration, a tracking change, a Search Console outage, or a major search update (all of these move impressions for reasons unrelated to the page).
Cannot claim: that a refresh will reverse a decline, or anything about how Google ranks pages.""")

INTENDED USE AND LIMITS

Who: SEO specialists and content managers deciding which pages to look at first this month.
For what: ordering a review list. It is decision-support, not an instruction to change anything.
Valid for: pages with at least 14 GSC-available days in each of the last two 28-day windows and at least 100 impressions in the latest window. Outside that, the model has seen nothing like the page.
Stops being valid: after a site migration, a tracking change, a Search Console outage, or a major search update (all of these move impressions for reasons unrelated to the page).
Cannot claim: that a refresh will reverse a decline, or anything about how Google ranks pages.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [6]:
print("""HUMAN REVIEW BEFORE ACTING
1. Is there a known external cause (seasonality, outage, migration)?
2. Is the drop in the page's impressions real in Search Console itself?
3. Does the reason code agree with what you see on the page?
4. Is the flagged page thin on data (thin_data)?

NEVER AUTOMATE
- Rewriting, merging, redirecting, or deleting a page from this list.
- Pausing a page because it is in watch_recovery.
- Any customer-facing claim that a page "will decline".""")

HUMAN REVIEW BEFORE ACTING
1. Is there a known external cause (seasonality, outage, migration)?
2. Is the drop in the page's impressions real in Search Console itself?
3. Does the reason code agree with what you see on the page?
4. Is the flagged page thin on data (thin_data)?

NEVER AUTOMATE
- Rewriting, merging, redirecting, or deleting a page from this list.
- Pausing a page because it is in watch_recovery.
- Any customer-facing claim that a page "will decline".


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [7]:
drift = pd.Series({f: L.psi(tr[f], test[f]) for f in L.FEATURES}).sort_values(ascending=False)
print("Population stability index, development data vs sealed test cutoff (0.1 small, 0.25 large):\n", drift.round(3).to_string())
triggers = {
    "macro_f1_below_best_baseline_on_newest_labelled_cutoff": True,
    "any_feature_psi_above": 0.25,
    "decline_base_rate_moves_by_more_than_points": 5,
    "share_of_pages_with_gsc_available_days_below": 0.6,
    "check_after_events": ["site migration", "tracking change", "Search Console outage", "major search update"],
    "cadence": "re-score monthly; re-fit when a trigger fires",
}
print("\nTriggers:", json.dumps(triggers, indent=1))

Population stability index, development data vs sealed test cutoff (0.1 small, 0.25 large):
 pos_B             0.554
ga4_cov_B         0.352
d_impr            0.275
d_pos             0.258
org_per_day_B     0.120
d_ctr             0.060
ctr_B             0.043
r7                0.028
cv_B              0.015
log_rate_B        0.015
eng_per_day_B     0.012
avail_share_B     0.001
active_share_B    0.000

Triggers: {
 "macro_f1_below_best_baseline_on_newest_labelled_cutoff": true,
 "any_feature_psi_above": 0.25,
 "decline_base_rate_moves_by_more_than_points": 5,
 "share_of_pages_with_gsc_available_days_below": 0.6,
 "check_after_events": [
  "site migration",
  "tracking change",
  "Search Console outage",
  "major search update"
 ],
 "cadence": "re-score monthly; re-fit when a trigger fires"
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
full_path = os.path.join(L.CACHE_DIR, "action_queue_full.csv")
queue[["client", "content", "action", "risk", "p_Decline", "p_Recovery", "p_Growth", "iB", "reasons"]].to_csv(full_path, index=False)
print("full queue (not for git):", full_path, "|", len(queue), "rows")

preview = [dict(rank=i + 1, action=r.action, reasons=r.reasons, p_decline=round(float(r.p_Decline), 2),
                impressions_28d_rounded=int(round(r.iB, -2)))
           for i, r in enumerate(queue.sort_values("risk", ascending=False).head(15).itertuples())]
out = dict(evidence=dict(precision_at_k=prec, base_decline=base_decline, decline_top_share=0.05, other=other),
           drift_top_feature=str(drift.index[0]), drift_top_psi=float(drift.iloc[0]), drift_table=drift.round(4).to_dict(),
           triggers=triggers, preview=preview)
json.dump(out, open("work/outputs/queue_evidence.json", "w"), indent=1)
print("wrote work/outputs/queue_evidence.json")

full queue (not for git): /content/capstone_cache/action_queue_full.csv | 9559 rows
wrote work/outputs/queue_evidence.json


In [9]:
fm = json.load(open("work/outputs/final_metrics.json"))
print("model macro F1:", round(fm["model_scores"]["macro_f1"], 3))
for k, v in fm["baselines"].items(): print(f"{k:<16}", round(v["macro_f1"], 3))
print("best baseline:", fm["best_baseline"], "| difference:", round(fm["diff_vs_best_baseline"], 3), "| 95% interval:", [round(x, 3) for x in fm["diff_ci95"]])

model macro F1: 0.383
majority         0.101
persistence      0.268
mean_reversion   0.22
recent_slope     0.306
momentum_tree    0.417
best baseline: momentum_tree | difference: -0.034 | 95% interval: [-0.067, -0.012]


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.